# Chatbot Application
This is to learn how to create a basic chatbot application using gradio and OpenAPI.
This specific chatbot is an itinerary planner. 

## Setup
This first section requires creating the chat style and the use case of the application.

In [1]:
from openai import OpenAI

system_prompt = """
Instruction: Create an itinerary based on the input location. 
    This may include planning transportation, housing, food, and various activities. 
    Some of the plans may already be set. 
    For example, the user may only want a day plan for a location they are already present in. 

Style: For descriptions, provide eloquent and brief explanation, but enough information
    For other fields, use key words to describe that field. Eg: 'Activity: Snowboarding, Location: Matterhorn Ski Paradise in Zermatt, Switzerland'
"""

def chat(message, history):
    client = OpenAI()
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}] 
    response = client.chat.completions.create(model="gpt-6-luna", messages=messages)
    return response.choices[0].message.content


## UI
the UI will be a simple chat interface using Gradio. 

In [ ]:

import gradio as gr
gr.ChatInterface(chat, title="Intelligent Itinerary Planning Assistant").launch(share=True, inbrowser=True,
inline=False)

/Users/budhilscomputer/repos/MyPythonProject/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


* Running on local URL:  http://127.0.0.1:7860


2026/09/29 14:34:27 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: i/o timeout



Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


# Example chat
<img src="images/kazakhstan_demo.png" alt="example_chat_kazakhstan" width="600">

This is a good output. However, there are only a few issues with this. 
In that, there is no formatting. 
As such, we need to format it using either clever prompt engineering, or pydantic for truly consistent output formats. 


In [1]:
from typing import Literal
from pydantic import BaseModel, Field
import os
from dotenv import load_dotenv

## SETUP
load_dotenv()  # loads OPENAI_API_KEY from .env into the environment
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - check your .env file"


# Langchain chat model formatting
from langchain.chat_models import init_chat_model

# Two models: a capable one for real work, a cheap one for high-volume scoring (JEV).
llm = init_chat_model("openai:gpt-6-luna")



## OUTPUT FORMATTING
Activity_Category = Literal["nature", "active", "wildlife", "food", "beverage", "festival", "sightseeing", "historical", "cultural", "relaxation", "wellness"]

class Activity_Entry(BaseModel):
    category: list[Activity_Category]
    activity_time: int = Field(description="Recommended allocated time for activity. recorded in minutes.")
    activity_name: str = Field(description="Name of the Activity")
    location: str = Field(description="Location of activity")
    description: str = Field(description="5-20 word description. Key words & type of person")

class Day(BaseModel):
    day_summary: str = Field(description="10-30 word summary of the day. Major highlights & overall vibe of the day")
    activites: list[Activity_Entry]
    transportation: str = Field(description="note modes of transportation between activities")

class Itinerary(BaseModel):
    trip_summary: str = Field(description="Overall description of the planned itinerary. What audience would like this itinerary? Is it fast paced, slow paced?")
    days: list[Day]

planner = llm.with_structured_output(Itinerary)



## The Tool Definition
After defininng the structured output, here is the tool's definition.

In [6]:
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage

system_prompt = """
Instruction: Create an itinerary based on the input location. 
    This may include planning transportation, housing, food, and various activities. 
    Some of the plans may already be set. 
    For example, the user may only want a day plan for a location they are already present in. 

Style: For descriptions, provide eloquent and brief explanation, but enough information. For other fields, use key words to describe that field. Eg: 'Activity: Snowboarding, Location: Matterhorn Ski Paradise in Zermatt, Switzerland'
"""

@tool
def run_planner(topic: str) -> Itinerary:
    """Create an itinerary for the given location."""
    return planner.invoke([
        SystemMessage(system_prompt),
        HumanMessage(topic),
    ])

## Chat Interface: Agent + Chat Interface
Now we tie it together with a full chat interface.
First we create a chat agent that can execute this tool

In [7]:
from langchain.agents import create_agent

agent = create_agent(
    model="openai:gpt-6-luna",
    tools=[run_planner],
    system_prompt=system_prompt,
)

In [10]:
def chat(message, history):
    messages = history + [
        {"role": "user", "content": message}
    ]
    result = agent.invoke({
        "messages": messages
    })
    return result["messages"][-1].content

import gradio as gr
gr.ChatInterface(
    chat,
    title="Intelligent Itinerary Planning Assistant"
).launch(
    share=True,
    inbrowser=True,
    inline=False,
)

* Running on local URL:  http://127.0.0.1:7862
* Running on public URL: https://f7a9ef9c2c573f0608.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Example Chat with Improved Output style
<img src="images/tokyo_itinerary_1.png" alt="tokyo_itinerary" width="600">
<img src="images/tokyo_itinerary_2.png" alt="tokyo_itinerary_json" width="600">

Here is a better formatted result!
However, the LLM still summarizes this. 
The only change we need to make is the following:
@tool(return_direct=True)


In [ ]:
@tool(return_direct=True)
def run_planner(topic: str) -> Itinerary:
    """Create an itinerary for the given location."""
    itinerary = planner.invoke([
        SystemMessage(content=system_prompt),
        HumanMessage(content=topic),
    ])

    # another change here, so it directly outputs json
    return itinerary.model_dump_json(indent=2)

# Remaining repeat code

agent_1 = create_agent(
    model="openai:gpt-6-luna",
    tools=[run_planner],
    system_prompt=system_prompt,
)

def chat_1(message, history):
    messages = history + [
        {"role": "user", "content": message}
    ]
    result = agent_1.invoke({
        "messages": messages
    })
    return result["messages"][-1].content

gr.ChatInterface(
    chat_1,
    title="More Intelligent Itinerary Planning Assistant"
).launch(
    share=True,
    inbrowser=True,
    inline=False,
)

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://b2afd70056b33c22bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Example Chat with Improved Output style
<img src="images/better_itinerary_json.png" alt="tokyo_itinerary" width="600">

```json
{
  "trip_summary": "Assumption: five days in Tokyo, moderate pace, first visit, and midrange budget; dates and interests are flexible. Stay in Shinjuku for transport and nightlife, Ueno for value and museums, or Ginza/Tokyo Station for polished stays and convenient rail access. Use a rechargeable IC card for local trains and buses. Customize the route, pace, and food choices to suit your dates and interests.",
  "days": [
    {
      "day_summary": "Get oriented in west Tokyo with a scenic crossing, lively neighborhoods, and an easy first taste of Japanese food.",
      "activites": [
        {
          "category": [
            "relaxation"
          ],
          "activity_time": 20,
          "activity_name": "Choose a lodging base",
          "location": "Shinjuku, Ueno, or Ginza/Tokyo Station",
          "description": "Shinjuku suits nightlife and connections; Ueno offers value; Ginza brings polished stays and central access."
        },
        {
          "category": [
            "sightseeing",
            "active"
          ],
          "activity_time": 90,
          "activity_name": "Explore Shibuya Crossing and surrounding streets",
          "location": "Shibuya",
          "description": "Watch the famous crossing, browse lively streets, and enjoy an energetic introduction to modern Tokyo."
        },
        {
          "category": [
            "food"
          ],
          "activity_time": 75,
          "activity_name": "Try a casual ramen or sushi dinner",
          "location": "Shibuya",
          "description": "Choose a neighborhood noodle shop or sushi counter for an approachable first meal."
        },
        {
          "category": [
            "sightseeing",
            "cultural"
          ],
          "activity_time": 90,
          "activity_name": "Walk through Harajuku and Omotesando",
          "location": "Harajuku and Omotesando",
          "description": "Contrast playful fashion lanes with elegant architecture and flagship stores; ideal for curious urban explorers."
        }
      ],
      "transportation": "From Haneda, take the train or airport limousine bus; from Narita, use the Narita Express or Skyliner plus a city connection. Use JR or Tokyo Metro between Shibuya and Harajuku; walk locally."
    },
    {
      "day_summary": "Discover Tokyo’s historic east, from temple gates and traditional shopping streets to museum-rich Ueno.",
      "activites": [
        {
          "category": [
            "historical",
            "cultural",
            "sightseeing"
          ],
          "activity_time": 120,
          "activity_name": "Visit Senso-ji and Nakamise-dori",
          "location": "Asakusa",
          "description": "Explore Tokyo’s best-known Buddhist temple and its lively traditional shopping approach."
        },
        {
          "category": [
            "food"
          ],
          "activity_time": 60,
          "activity_name": "Lunch on soba, tempura, or street snacks",
          "location": "Asakusa",
          "description": "Sample local classics near the temple; convenient for traditional-food fans and relaxed diners."
        },
        {
          "category": [
            "cultural",
            "sightseeing"
          ],
          "activity_time": 90,
          "activity_name": "Browse Ueno Park and one museum",
          "location": "Ueno Park",
          "description": "Choose the Tokyo National Museum or another park museum for a focused cultural visit."
        },
        {
          "category": [
            "food",
            "sightseeing"
          ],
          "activity_time": 60,
          "activity_name": "Explore Ameyoko market street",
          "location": "Ueno",
          "description": "Browse busy market lanes and casual eateries; a lively stop for snacks and people-watching."
        }
      ],
      "transportation": "Take Tokyo Metro Ginza Line between Asakusa and Ueno, or use a taxi for convenience. Walk between Ueno Park and Ameyoko."
    },
    {
      "day_summary": "Balance peaceful shrine grounds with city views, contemporary shopping, and the bright buzz of Shinjuku.",
      "activites": [
        {
          "category": [
            "nature",
            "cultural",
            "historical"
          ],
          "activity_time": 90,
          "activity_name": "Walk the Meiji Jingu shrine grounds",
          "location": "Meiji Jingu, Shibuya",
          "description": "Follow forested paths to a major Shinto shrine, a restorative pause beside the city’s busiest districts."
        },
        {
          "category": [
            "food"
          ],
          "activity_time": 60,
          "activity_name": "Lunch around Shinjuku Station",
          "location": "Shinjuku",
          "description": "Pick from department-store food halls or casual restaurants for variety and easy seating."
        },
        {
          "category": [
            "sightseeing"
          ],
          "activity_time": 75,
          "activity_name": "See the city from a free observation deck",
          "location": "Tokyo Metropolitan Government Building, Shinjuku",
          "description": "Take in broad city views from the public observatory; check opening hours and visibility beforehand."
        },
        {
          "category": [
            "food",
            "cultural"
          ],
          "activity_time": 90,
          "activity_name": "Dinner in Omoide Yokocho or Shinjuku Golden Gai",
          "location": "Shinjuku",
          "description": "Discover compact eateries and atmospheric lanes; choose a welcoming spot and check seating or cover charges."
        }
      ],
      "transportation": "Use JR Yamanote Line or Tokyo Metro to reach Shinjuku from Harajuku. The shrine, station area, and evening lanes are walkable, though the station is extensive."
    },
    {
      "day_summary": "Enjoy Tokyo’s food culture and refined central districts, with time for shopping, gardens, or an unhurried café break.",
      "activites": [
        {
          "category": [
            "food"
          ],
          "activity_time": 90,
          "activity_name": "Browse Tsukiji Outer Market",
          "location": "Tsukiji",
          "description": "Taste seafood, tamagoyaki, and market snacks; arrive earlier for the liveliest stalls."
        },
        {
          "category": [
            "sightseeing",
            "relaxation"
          ],
          "activity_time": 90,
          "activity_name": "Stroll Hamarikyu Gardens",
          "location": "Hamarikyu Gardens",
          "description": "Find a tranquil garden landscape and teahouse beside the modern waterfront skyline."
        },
        {
          "category": [
            "sightseeing",
            "cultural"
          ],
          "activity_time": 120,
          "activity_name": "Explore Ginza and the Tokyo Station area",
          "location": "Ginza and Marunouchi",
          "description": "Admire department stores, architecture, and station-front plazas; appealing to design and shopping enthusiasts."
        },
        {
          "category": [
            "food"
          ],
          "activity_time": 90,
          "activity_name": "Dinner at an izakaya or sushi restaurant",
          "location": "Ginza or Yurakucho",
          "description": "Enjoy Japanese small plates or sushi; reserve popular restaurants, especially on weekends."
        }
      ],
      "transportation": "Take Tokyo Metro to Tsukiji and walk to Hamarikyu. Continue by taxi or walk toward Ginza; use JR or Metro from Yurakucho or Tokyo Station."
    },
    {
      "day_summary": "Choose a final-day neighborhood that matches your interests, then leave time for souvenirs and an easy airport connection.",
      "activites": [
        {
          "category": [
            "cultural",
            "sightseeing"
          ],
          "activity_time": 120,
          "activity_name": "Explore Akihabara’s electronics and pop-culture shops",
          "location": "Akihabara",
          "description": "Browse game, anime, and electronics stores; especially fun for pop-culture and gadget enthusiasts."
        },
        {
          "category": [
            "food"
          ],
          "activity_time": 60,
          "activity_name": "Lunch at a neighborhood curry or noodle shop",
          "location": "Akihabara or nearby Kanda",
          "description": "Choose a quick, satisfying local meal that leaves room for afternoon exploring."
        },
        {
          "category": [
            "sightseeing",
            "relaxation"
          ],
          "activity_time": 150,
          "activity_name": "Choose a flexible final stop",
          "location": "Odaiba waterfront, teamLab Planets in Toyosu, or a neighborhood of choice",
          "description": "Pick waterfront views, an immersive art visit, or extra time in a favorite district; book timed-entry attractions ahead."
        },
        {
          "category": [
            "food",
            "relaxation"
          ],
          "activity_time": 60,
          "activity_name": "Pick up gifts and an early final meal",
          "location": "Tokyo Station or lodging neighborhood",
          "description": "Shop for packaged snacks and souvenirs, then keep dinner simple before travel or departure."
        }
      ],
      "transportation": "Use JR to Akihabara. Reach Odaiba via the Yurikamome Line and teamLab Planets via the Yurikamome or Tokyo Metro. Allow generous time for luggage collection and airport transit."
    }
  ]
}
```



